In [14]:
!pip install -q langchain langchain-community langchain-groq langchain-huggingface langchain-chroma langchain-text-splitters sentence-transformers pypdf

In [15]:
from langchain_community.document_loaders import (
    DirectoryLoader,
    PyPDFLoader,
    TextLoader
)

def load_documents():

    pdf_loader = DirectoryLoader(
        "/content/",
        glob="**/*.pdf",
        loader_cls=PyPDFLoader
    )
    pdf_docs = pdf_loader.load()

    documents = pdf_docs

    return documents

In [16]:
docs = load_documents()

print(f"Loaded {len(docs)} documents")

print(docs[0].page_content[:500])

Loaded 1 documents
Mohamed Shereef CH
♂phone+91-9061139031✉mhdshareefch@gmail.com/gl⌢bemhdshareef-portfolio.vercel.app/githubgithub.com/spigelspike
Summary
AI/ML Engineer with hands-on experience building, fine-tuning, and deploying Deep Learning architectures, production RAG
pipelines, and Multimodal GenAI systems. Proficient in Python, PyTorch, Hugging Face, vector databases (ChromaDB), and
high-throughput LLM serving (Groq, Gemini). Strong foundation in medical computer vision, dual-stage OCR pipelines, RAG
eva


In [52]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def split_documents(docs):
    # Increased chunk size to ensure related items stay together
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=200
    )

    chunks = splitter.split_documents(docs)

    return chunks

In [41]:
chunks = split_documents(docs)

print(f"Original documents: {len(docs)}")
print(f"Chunks created: {len(chunks)}")

print("\nFirst chunk:")
print(chunks[0].page_content)

Original documents: 1
Chunks created: 8

First chunk:
Mohamed Shereef CH
♂phone+91-9061139031✉mhdshareefch@gmail.com/gl⌢bemhdshareef-portfolio.vercel.app/githubgithub.com/spigelspike
Summary
AI/ML Engineer with hands-on experience building, fine-tuning, and deploying Deep Learning architectures, production RAG
pipelines, and Multimodal GenAI systems. Proficient in Python, PyTorch, Hugging Face, vector databases (ChromaDB), and
high-throughput LLM serving (Groq, Gemini). Strong foundation in medical computer vision, dual-stage OCR pipelines, RAG
evaluation frameworks (RAGAS), and asynchronous backend engineering.
Education
• Bachelor of Technology in Information Technology05/2026
MEA Engineering College, Kerala
Technical Skills


In [42]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma


embeddings = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2"
)


vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings
)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [53]:
query = "What are the technical frameworks, libraries, and tools mentioned?"

# Increase k to 7 to provide a wider context window for the LLM
results = vector_store.similarity_search(
    query,
    k=7
)

In [44]:
import os
from google.colab import userdata

os.environ["GROQ_API_KEY"] = userdata.get("groq_api")

In [45]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0
)

In [51]:
query = input("Enter your question: ")

context = "\n\n".join(
    doc.page_content for doc in results
)

# Enhanced prompt for better extraction
prompt = f"""
You are an expert recruiter assistant. Answer the question comprehensively based ONLY on the provided context.
If the information is not in the context, say you don't know.

Context:
{context}

Question:
{query}

Answer (use bullet points if listing multiple items):
"""

response = llm.invoke(prompt)

print("\n--- Analysis ---\n")
print(response.content)

Enter your question: what are the framework that he is  familair with

--- Analysis ---

**Frameworks he is familiar with**

- **Deep Learning / Computer Vision**
  - PyTorch  
  - Torchvision  
  - EfficientNet  
  - Grad‑CAM  
  - OpenCV  
  - Hugging Face Transformers  
  - scikit‑learn  

- **Generative AI & Retrieval‑Augmented Generation (RAG)**
  - RAG Architecture  
  - Sentence‑Transformers  
  - ChromaDB (vector store)  
  - Groq (LLaMA‑3)  
  - Gemini 2.5  
  - Flash Vision  
  - Ollama  
  - RAGAS (evaluation)  

- **Backend & API Development**
  - FastAPI (with Uvicorn)  
  - Node.js  

- **Cloud / DevOps**
  - Docker (and Docker Compose)  

These are the primary frameworks mentioned in the provided context.
